In [4]:
import json
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


# ==========================================
# 1. 定义 State
# ==========================================
class EmployeeState(TypedDict):
    employee_id: str
    employee: dict | None
    permission_granted: bool
    leader: dict | None
    error: str | None
    logs: Annotated[list[str], operator.add]


# ==========================================
# 2. 准备模拟数据
# ==========================================
EMPLOYEE_DB = {
    "E1001": {
        "id": "E1001",
        "name": "张三",
        "department": "研发部",
        "leader_id": "E9001",
    },
    "E1002": {
        "id": "E1002",
        "name": "李四",
        "department": "设计部",
        "leader_id": "E9001",
    },
    "E9001": {
        "id": "E9001",
        "name": "李经理",
        "department": "研发部",
        "leader_id": None,
    },
}


# ==========================================
# 3. 编写 Node (State -> Partial State)
# ==========================================
def query_employee(state: EmployeeState) -> dict:
    employee_id = state["employee_id"]
    employee = EMPLOYEE_DB.get(employee_id)

    if employee is None:
        return {
            "employee": None,
            "error": f"员工 {employee_id} 不存在",
            "logs": [f"员工查询失败：{employee_id}"],
        }

    return {
        "employee": employee,
        "error": None,
        "logs": [f"员工查询成功：{employee['name']}"],
    }


def check_permission(state: EmployeeState) -> dict:
    employee = state["employee"]

    if employee is None:
        return {
            "permission_granted": False,
            "error": "员工数据不存在，无法检查权限",
            "logs": ["权限检查失败"],
        }

    # 只有 E1001 拥有特权账号权限
    granted = employee["id"] == "E1001"

    return {
        "permission_granted": granted,
        "error": None if granted else "没有员工查询权限",
        "logs": ["权限验证通过" if granted else "权限验证失败"],
    }


def query_leader(state: EmployeeState) -> dict:
    employee = state["employee"]

    if employee is None:
        return {
            "leader": None,
            "error": "员工信息不存在",
            "logs": ["无法查询领导"],
        }

    leader_id = employee.get("leader_id")

    if leader_id is None:
        return {
            "leader": None,
            "error": None,
            "logs": ["该员工没有直属领导"],
        }

    leader = EMPLOYEE_DB.get(leader_id)

    return {
        "leader": leader,
        "error": None,
        "logs": [
            f"查询到直属领导：{leader['name']}" if leader else "直属领导不存在"
        ],
    }


def employee_not_found(state: EmployeeState) -> dict:
    return {"logs": ["流程结束：员工不存在"]}


def permission_denied(state: EmployeeState) -> dict:
    return {"logs": ["流程结束：权限不足"]}


# ==========================================
# 4. 编写 Router
# ==========================================
def route_after_employee(
    state: EmployeeState,
) -> Literal["check_permission", "employee_not_found"]:
    if state["employee"] is None:
        return "employee_not_found"
    return "check_permission"


def route_after_permission(
    state: EmployeeState,
) -> Literal["query_leader", "permission_denied"]:
    if state["permission_granted"]:
        return "query_leader"
    return "permission_denied"


# ==========================================
# 5. 构建 Graph
# ==========================================
def build_graph():
    builder = StateGraph(EmployeeState)

    # 注册节点
    builder.add_node("query_employee", query_employee)
    builder.add_node("check_permission", check_permission)
    builder.add_node("query_leader", query_leader)
    builder.add_node("employee_not_found", employee_not_found)
    builder.add_node("permission_denied", permission_denied)

    # 注册边
    builder.add_edge(START, "query_employee")

    builder.add_conditional_edges(
        "query_employee",
        route_after_employee,
        {
            "check_permission": "check_permission",
            "employee_not_found": "employee_not_found",
        },
    )

    builder.add_conditional_edges(
        "check_permission",
        route_after_permission,
        {
            "query_leader": "query_leader",
            "permission_denied": "permission_denied",
        },
    )

    builder.add_edge("query_leader", END)
    builder.add_edge("employee_not_found", END)
    builder.add_edge("permission_denied", END)

    # 配置内存 Checkpointer
    checkpointer = InMemorySaver()
    return builder.compile(checkpointer=checkpointer)


# ==========================================
# 6. 测试与运行
# ==========================================
def run_scenario(graph, employee_id: str, thread_id: str, title: str):
    print("=" * 60)
    print(f"▶ {title} [Target ID: {employee_id}]")
    print("=" * 60)

    initial_state: EmployeeState = {
        "employee_id": employee_id,
        "employee": None,
        "permission_granted": False,
        "leader": None,
        "error": None,
        "logs": [],
    }

    config = {"configurable": {"thread_id": thread_id}}

    result = graph.invoke(initial_state, config=config)

    print("【最终 State】:")
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print("\n【执行 Trace Logs】:")
    for idx, log in enumerate(result["logs"], 1):
        print(f"  {idx}. {log}")
    print("\n")


if __name__ == "__main__":
    app = build_graph()

    # 测试场景 1：员工存在且有权限
    run_scenario(app, "E1001", "thread-001", "场景 1：员工存在且有权限")

    # 测试场景 2：员工不存在
    run_scenario(app, "E9999", "thread-002", "场景 2：员工不存在")

    # 测试场景 3：员工存在但无权限
    run_scenario(app, "E1002", "thread-003", "场景 3：员工存在但无权限")

▶ 场景 1：员工存在且有权限 [Target ID: E1001]
【最终 State】:
{
  "employee_id": "E1001",
  "employee": {
    "id": "E1001",
    "name": "张三",
    "department": "研发部",
    "leader_id": "E9001"
  },
  "permission_granted": true,
  "leader": {
    "id": "E9001",
    "name": "李经理",
    "department": "研发部",
    "leader_id": null
  },
  "error": null,
  "logs": [
    "员工查询成功：张三",
    "权限验证通过",
    "查询到直属领导：李经理"
  ]
}

【执行 Trace Logs】:
  1. 员工查询成功：张三
  2. 权限验证通过
  3. 查询到直属领导：李经理


▶ 场景 2：员工不存在 [Target ID: E9999]
【最终 State】:
{
  "employee_id": "E9999",
  "employee": null,
  "permission_granted": false,
  "leader": null,
  "error": "员工 E9999 不存在",
  "logs": [
    "员工查询失败：E9999",
    "流程结束：员工不存在"
  ]
}

【执行 Trace Logs】:
  1. 员工查询失败：E9999
  2. 流程结束：员工不存在


▶ 场景 3：员工存在但无权限 [Target ID: E1002]
【最终 State】:
{
  "employee_id": "E1002",
  "employee": {
    "id": "E1002",
    "name": "李四",
    "department": "设计部",
    "leader_id": "E9001"
  },
  "permission_granted": false,
  "leader": null,
  "error": "没有员工查询权限",
  "